# 1. Análise exploratória dos preços da cesta básica

## 1.1 Introdução e questão de pesquisa
É possível antecipar uma alta **superior a 2% no mês seguinte**, para um produto e supermercado, usando preços históricos? Esta etapa descreve dados e qualidade; não estima desempenho preditivo.

## 2. Descrição e leitura da fonte
O repositório contém uma planilha XLSX, não o CSV originalmente esperado. A aba disponível é lida diretamente, sem alterar a fonte. `Mes_Ano_Data` é o mês de referência; `Data_Coleta` pode estar ausente e não é inventada.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from basket_price_prediction.config import ROOT, DATASET, SPLITS, FEATURES
from basket_price_prediction.data import load_data
from basket_price_prediction.preprocessing import prepare_monthly
from basket_price_prediction.features import build_supervised, temporal_split
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
raw = load_data()
display(raw.head(10))
print("Dimensões:", raw.shape)
display(pd.DataFrame({"tipo":raw.dtypes.astype(str), "ausentes":raw.isna().sum(), "distintos":raw.nunique()}))

## 3. Cobertura e integridade
Inspecione observações repetidas, datas ausentes, valores inválidos e possíveis divergências entre mês de coleta e referência. Contagens de motivos podem se sobrepor.

In [ ]:
monthly, cleaning = prepare_monthly(raw)
print("Período:", raw.Mes_Ano_Data.min(), "a", raw.Mes_Ano_Data.max())
display(pd.Series(cleaning, name="valor").to_frame())
display(raw[["Produto", "Marca", "Quantidade"]].nunique().to_frame("valores distintos"))
display(raw.groupby("Produto").agg(registros=("Preco","size"), marcas=("Marca","nunique"), quantidades=("Quantidade","nunique")).sort_values("registros",ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,4))
raw.groupby("Mes_Ano_Data").size().plot(ax=axes[0]); axes[0].set(title="Registros por mês de referência", xlabel="Mês", ylabel="Registros")
sns.heatmap(raw.isna().sample(min(1000,len(raw)),random_state=42), cbar=False, ax=axes[1]); axes[1].set(title="Dados ausentes — amostra de 1.000 linhas", xlabel="Coluna", ylabel="Linha amostrada")
plt.tight_layout(); plt.show()

## 4. Padronização e definição de série
`PPK` é comparado com preço dividido por quantidade; a unidade física de cada produto não está documentada. Portanto não se afirma que seja R$/kg. A variável escolhida é o preço da embalagem original, mantida constante por supermercado, produto, marca e quantidade. Marca ausente é uma categoria explícita, sujeita a misturar marcas desconhecidas. Duplicatas exatas são removidas e registros distintos do mesmo mês usam mediana. Nenhum extremo é removido automaticamente.

In [ ]:
display(raw.groupby("Produto").agg(quantidades=("Quantidade",lambda x: sorted(x.unique().tolist())), marcas=("Marca","nunique")).head(30))
print("Divergências PPK vs preço/quantidade > R$ 0,01:",cleaning["ppk_discrepancies_over_0_01"])
print("Duplicatas exatas:",cleaning["exact_duplicates"],"; registros agregados além do primeiro:",cleaning["aggregated_extra_records"])
print("Descrições originais:", sorted(raw.Produto.dropna().unique()))

## 5. Distribuição de preços e diferenças entre estabelecimentos
Os preços de produtos e tamanhos distintos não são diretamente comparáveis; os gráficos globais são apenas diagnósticos.

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(14,4))
sns.histplot(raw, x="Preco", bins=80, ax=axes[0]); axes[0].set(xlabel="Preço por embalagem (R$)",ylabel="Registros",title="Distribuição dos preços")
top=raw.Produto.value_counts().head(1).index[0]
sub=raw[raw.Produto.eq(top)].copy(); stores=sub.Estab.value_counts().head(12).index
sns.boxplot(data=sub[sub.Estab.isin(stores)], x="Estab", y="Preco", ax=axes[1],showfliers=False); axes[1].tick_params(axis="x",rotation=45); axes[1].set(title=f"Preço de {top} por supermercado (embalagens variadas)",xlabel="Supermercado",ylabel="Preço por embalagem (R$)")
plt.tight_layout(); plt.show()

## 6. Tendências e composição
As medianas abaixo misturam lojas e marcas: descrevem a amostra observada, não uma cesta fixa ou inflação oficial. Não há coluna de categoria confiável; a distribuição é exibida por produto.

In [ ]:
top_products=raw.Produto.value_counts().head(6).index
trend=raw[raw.Produto.isin(top_products)].groupby(["Mes_Ano_Data","Produto"],as_index=False).Preco.median()
fig, axes=plt.subplots(1,2,figsize=(16,5))
sns.lineplot(data=trend,x="Mes_Ano_Data",y="Preco",hue="Produto",ax=axes[0]); axes[0].set(title="Mediana mensal de produtos selecionados",xlabel="Mês",ylabel="Preço por embalagem (R$)")
raw.Produto.value_counts().head(20).sort_values().plot.barh(ax=axes[1]); axes[1].set(title="Distribuição por produto",xlabel="Registros",ylabel="Produto")
plt.tight_layout(); plt.show()

## 7. Variações, volatilidade e disponibilidade
Mudanças são calculadas apenas entre meses consecutivos da mesma série. Valores extremos podem ser erros, promoções ou alterações reais e precisam de auditoria na origem. Volatilidade é o desvio padrão das variações mensais observadas.

In [ ]:
from basket_price_prediction.config import ENTITY
m=monthly.sort_values(ENTITY+["month"]).copy(); g=m.groupby(ENTITY,observed=True)
prev=g.price_current.shift(1); prev_month=g.month.shift(1)
consecutive=(m.month.dt.to_period("M").astype("int64")-prev_month.dt.to_period("M").astype("int64")).eq(1)
m["change"]= (m.price_current/prev-1).where(consecutive)
print("Maiores variações absolutas — verificar, não excluir:")
display(m.loc[m.change.abs().nlargest(15).index, ENTITY+["month","price_current","change"]])
fig,axes=plt.subplots(1,2,figsize=(14,4))
sns.histplot(m.change.dropna().clip(-1,1)*100,bins=80,ax=axes[0]); axes[0].set(title="Variação mensal (visualização limitada a ±100%)",xlabel="Variação (%)",ylabel="Observações")
vol=m.groupby("product").change.agg(["std","count"]).query("count >= 30").sort_values("std").tail(15)
vol["std"].mul(100).plot.barh(ax=axes[1]); axes[1].set(title="Volatilidade das variações por produto",xlabel="Desvio padrão (%)",ylabel="Produto")
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,4))
availability=m.groupby("month").size(); availability.plot(ax=axes[0]); axes[0].set(title="Séries produto–loja–marca–embalagem por mês",xlabel="Mês",ylabel="Séries observadas")
direction=pd.cut(m.change,[-np.inf,-1e-12,1e-12,np.inf],labels=["Queda","Estável","Alta"]).value_counts().reindex(["Queda","Estável","Alta"])
direction.plot.bar(ax=axes[1]); axes[1].set(title="Direção mensal dos preços",xlabel="Direção",ylabel="Observações")
plt.tight_layout(); plt.show()

## 8. Distribuição inicial do alvo
O alvo usa o preço observado no mês calendário seguinte; sem ele não há rótulo. A amostra abaixo também exige dois meses prévios consecutivos.

In [ ]:
supervised, feature_report=build_supervised(monthly)
display(pd.Series(feature_report,name="observações").to_frame())
display(supervised.target.value_counts().rename(index={0:"Sem alta >2%",1:"Alta >2%"}).to_frame("n"))
display(supervised.groupby(supervised.target_month.dt.to_period("M")).target.agg(["size","mean"]).rename(columns={"size":"n", "mean":"fração de altas"}).tail(12))

## 9. Conclusões exploratórias
A elegibilidade é afetada por marcas desconhecidas, séries interrompidas e pela exigência de histórico; observações elegíveis podem não representar todas as compras. Diferenças de embalagem, descrições e metodologia de coleta dificultam comparações. Os dados possuem meses e classes suficientes para uma avaliação supervisionada temporal, sem garantir estabilidade ou causalidade. Confira as contagens acima e os resultados independentes no notebook 3.

In [ ]:
display(Markdown(f"Foram observados **{len(raw):,}** registros originais, **{cleaning['monthly_series']:,}** séries mensais distintas e **{len(supervised):,}** exemplos elegíveis; a fração de altas é **{supervised.target.mean():.1%}**."))